# HW4 — Features and a first Ridge model
**Author:** Mauricio Gutierrez

Use this notebook after `01_data_preparation.ipynb`.
The first notebook checks and cleans the database. This notebook creates model
inputs and tests a first return model on the small class dataset.

```text
Cleaned database -> Past/current prices -> Five features -> Dated candidates
                                                              |
Completed historical targets -> Training rows -> Fit pipeline  |
                                                    |         |
                                                    v         v
                                                  Return forecasts
                                                        |
Future prices -> Separate return targets ----------------+-> Model checks
```

The output is a return forecast for each eligible stock and signal date.
Stock selection, purchase amounts, and portfolio accounting belong in the later
backtester. A model error score is not a portfolio return.

The five features and `alpha=100` are starting settings. They are not optimized.
Financial features and DCF are outside this first experiment.

> **Development step, class data (30 stocks).** First model: ridge with five price features. The final strategy is defined in `run_final_test.py` and reported in `07_final_test_report.ipynb`.

## 1. Import the reusable functions
Run the cells in order. Select the repository `.venv` as the Python kernel.
The module `feature_engineering.py` contains reusable functions. Importing it
does not run the cleaner or fit a model.

In [1]:
from pathlib import Path
import sys
import json
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

# Support a kernel started in this folder or anywhere inside the repository.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "Lectures/modeling/course_data.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Start the kernel inside the course repository.")
HW_DIR = ROOT / "Homeworks/HW4/Mauricio_Gutierrez"
if str(HW_DIR) not in sys.path:
    sys.path.insert(0, str(HW_DIR))

from feature_engineering import (
    PRICE_FEATURES, load_cleaned_data, price_features, apply_membership,
    make_labels, make_ridge_pipeline, annual_split, prediction_metrics,
)

## 2. Set the experiment
Read the cleaned class database. Keep the final test closed.
Use all earlier prices for feature lookbacks. Start training signals in 2010.
Validate on 2013, 2014, and 2015. Refit before each validation year with only
completed historical targets. This is an expanding training window.

The small database stops in 2015. Late-2015 targets cannot finish here.
Keep those forecasts. Exclude only their missing outcomes from error scores.
The final-test retraining rule remains a separate decision.

In [2]:
DATABASE = HW_DIR / "outputs/data_cleaning/equities_class/equities_class_cleaned.duckdb"
DATA_END = "2015-12-31"
TRAIN_START = "2010-01-01"
VALIDATION_YEARS = (2013, 2014, 2015)
FEATURES = PRICE_FEATURES.copy()
RIDGE_ALPHA = 100.0
HOLDING_DAYS = 90  # Calendar days, measured from the next-session entry close.
MAX_EXIT_DELAY = 0  # No target when the required exit session has no stock quote.
SAVE_OUTPUTS = True
OUTPUT_DIR = HW_DIR / "outputs/ridge_baseline/equities_class"

def file_digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

database_hash_before = file_digest(DATABASE)
prices, calendar, membership = load_cleaned_data(DATABASE, end=DATA_END)
display(pd.DataFrame({
    "Input": ["prices", "calendar", "universe_membership"],
    "Rows": [len(prices), len(calendar), len(membership)],
    "Purpose": ["Adjusted closes", "Market-session alignment", "Dated candidate lists"],
}))
print(f"Stocks: {prices.symbol.nunique()} | Prices: {prices.date.min().date()} to {prices.date.max().date()}")

Stocks: 30 | Prices: 2008-01-02 to 2015-12-31


,Input,Rows,Purpose
0,prices,59286,Adjusted closes
1,calendar,2015,Market-session alignment
2,universe_membership,937,Dated candidate lists


## 3. Understand the inputs
The cleaned database contains several tables. This experiment reads three.

| Table | Fields used | Meaning |
|---|---|---|
| `prices` | `symbol`, `date`, `adj_close` | Stock identifier, session date, adjusted close |
| `calendar` | `date` | The ordered market sessions |
| `universe_membership` | Stock, quarter, selection date, rank | The candidate list known for an entry quarter |

`adj_close` includes the source's price adjustments. Use it consistently for
features and targets. Do not add dividends again.
No feature uses a current active-status flag, a future return, or a cleaning audit flag.

Use sessions for the feature windows. Use calendar days for the holding period.
Reindex each stock to the market calendar. Leave missing prices missing.
Do not treat the previous observed quote as yesterday's quote across a gap.

In [3]:
display(prices.head())
display(membership.head())

,symbol,date,adj_close
0,AAPL,2008-01-02,5.83
1,AAPL,2008-01-03,5.83
2,AAPL,2008-01-04,5.39
3,AAPL,2008-01-07,5.31
4,AAPL,2008-01-08,5.13


,symbol,quarter_start,quarter_end,selection_date,universe_rank
0,XOM,2008-01-01,2008-03-31,2007-12-31,1
1,POZN,2008-01-01,2008-03-31,2007-12-31,3
2,MSFT,2008-01-01,2008-03-31,2007-12-31,4
3,PEGY,2008-01-01,2008-03-31,2007-12-31,5
4,CCZ,2008-01-01,2008-03-31,2007-12-31,7


## 4. Build the five starting features
Let `P(t)` be the adjusted close on signal session `t`.
Every input below is available after that session closes.

| Feature | Calculation | Question it describes |
|---|---|---|
| `momentum_21` | `P(t) / P(t-21) - 1` | How much did the stock move over 21 sessions? |
| `momentum_63` | `P(t) / P(t-63) - 1` | How much did the stock move over 63 sessions? |
| `momentum_252_21` | `P(t-21) / P(t-252) - 1` | What was the longer return, excluding the latest 21 sessions? |
| `volatility_63` | Sample standard deviation of 63 daily returns × √252 | How variable were recent daily returns? |
| `drawdown_252` | `P(t) / max(last 252 session prices) - 1` | How far is the stock below its one-year high? |

Returns use decimal units: `0.10` means 10%. Drawdown is zero or negative.
This drawdown feature is not an all-time-high filter. None of these features
forces a purchase or sale. Ridge learns their coefficients from training data.

Require 63 consecutive valid daily returns for price eligibility. This is the
lecture's starting rule. Longer-window features can still be missing.
The pipeline fills those feature values from training medians. It never fills prices.

In [4]:
feature_panel = price_features(prices, calendar)
feature_panel = apply_membership(feature_panel, membership, calendar)
development = feature_panel.loc[feature_panel.date.ge(TRAIN_START)]
eligible_features = development.loc[development.eligible, FEATURES]
display(pd.DataFrame({
    "missing_percent_among_eligible": eligible_features.isna().mean().mul(100),
    "median": eligible_features.median(),
    "minimum": eligible_features.min(),
    "maximum": eligible_features.max(),
}))
print(f"Development rows: {len(development):,}; eligible signal rows: {development.eligible.sum():,}")
display(development.loc[development.eligible, ["date", "symbol", *FEATURES]].head())

Development rows: 44,385; eligible signal rows: 43,381


,missing_percent_among_eligible,median,minimum,maximum
momentum_21,0.0,0.008692,-0.652755,2.346154
momentum_63,0.0,0.025861,-0.676012,4.364706
momentum_252_21,0.0,0.118943,-0.798658,6.000000
volatility_63,0.0,0.199228,0.062897,4.067115
drawdown_252,0.0,-0.065357,-0.809524,0.000000


,date,symbol,momentum_21,momentum_63,momentum_252_21,volatility_63,drawdown_252
14901,2010-01-04,AAPL,0.088435,0.157324,1.161765,0.267796,0.000000
14902,2010-01-04,AMT,0.046028,0.218957,0.376139,0.244930,-0.000301
14903,2010-01-04,APDN,-0.111842,-0.413043,0.604444,0.914055,-0.652062
14904,2010-01-04,BESS,-0.111111,0.500000,1.700000,4.067115,-0.384615
14905,2010-01-04,BKNG,0.027209,0.371661,1.835604,0.447108,-0.023023


## 5. Add separate return targets
The signal uses today's close. The entry uses the next market session's close.
Measure the 90 calendar days from entry. Move a closed-market maturity date to
the next market session.

```text
Signal close -> Next-session entry close -> Entry + 90 calendar days
                                                     |
                                                     v
                                         First market session on/after maturity
                                                     |
                                                     v
                                  Target = exit adjusted price / entry adjusted price - 1
```

The target is a gross holding-period return. It is neither annualized nor net of fees.
If the entry or required exit quote is missing, leave the target missing.
This experiment does not delay exits to search for later quotes.
Notebook 03 stops the backtest if a held stock has no required quote.

Future prices appear only in target and audit columns. `FEATURES` is an explicit
allowlist for model inputs. Target availability must not decide who receives a forecast.

In [5]:
panel = make_labels(
    feature_panel, prices, calendar,
    holding_days=HOLDING_DAYS, max_exit_delay=MAX_EXIT_DELAY,
)
assert panel.loc[panel.eligible, FEATURES].equals(feature_panel.loc[feature_panel.eligible, FEATURES])
example = panel.loc[panel.eligible & panel.date.ge(TRAIN_START) & panel.target.notna()].head(5)
display(example[["symbol", "date", "entry_date", "scheduled_exit", "label_end", "entry_price", "exit_price", "target"]])

,symbol,date,entry_date,scheduled_exit,label_end,entry_price,exit_price,target
14901,AAPL,2010-01-04,2010-01-05,2010-04-05,2010-04-05,6.42,7.14,0.112150
14902,AMT,2010-01-04,2010-01-05,2010-04-05,2010-04-05,33.73,33.08,-0.019271
14903,APDN,2010-01-04,2010-01-05,2010-04-05,2010-04-05,2520000.15,2520000.15,0.000000
14904,BESS,2010-01-04,2010-01-05,2010-04-05,2010-04-05,168.00,119.00,-0.291667
14905,BKNG,2010-01-04,2010-01-05,2010-04-05,2010-04-05,224.94,255.81,0.137237


## 6. Check the time boundary
Change every price after a cutoff. Earlier features must stay identical.
This checks that feature calculations cannot read future prices.
The module's separate test file also checks gaps, quarter boundaries, target
dates, incomplete outcomes, and fitted preprocessing.

In [6]:
cutoff = pd.Timestamp("2013-06-28")
changed_prices = prices.copy()
changed_prices.loc[changed_prices.date.gt(cutoff), "adj_close"] *= 7.0
changed_features = price_features(changed_prices, calendar)
original_features = price_features(prices, calendar)
pd.testing.assert_frame_equal(
    original_features.loc[original_features.date.le(cutoff), ["date", "symbol", *FEATURES, "eligible"]],
    changed_features.loc[changed_features.date.le(cutoff), ["date", "symbol", *FEATURES, "eligible"]],
)
print("PASS: changing future prices does not change earlier features.")

PASS: changing future prices does not change earlier features.


## 7. Fit the pipeline and forecast each validation year
Ridge fits a linear relationship between features and the return target.
Its penalty reduces large coefficients. `alpha` controls that penalty.

```text
Training features -> Fit 1%/99% clipping bounds -> Fit medians -> Fit scaling -> Fit Ridge
Validation features -> Apply those saved bounds -> Apply medians -> Apply scaling -> Predict
```

The clipping step limits extreme inputs. The imputer adds missing-value indicators.
The scaler uses training means and standard deviations.
No step learns from validation features during `predict`.

Before each fit, remove training rows whose target has not ended.
For example, a December signal with a March exit is unavailable to a January fit.
Compare Ridge with a constant forecast equal to the training target mean.
Do not choose settings from the final test.

In [7]:
predictions = []
split_records = []
models = {}
for year in VALIDATION_YEARS:
    train, signals = annual_split(panel, year, train_start=TRAIN_START)
    assert train.label_end.max() < signals.date.min()
    model = make_ridge_pipeline(alpha=RIDGE_ALPHA, feature_names=FEATURES)
    model.fit(train[FEATURES], train.target)
    result = signals[["date", "symbol", "entry_date", "scheduled_exit", "label_end", "target"]].copy()
    result["prediction"] = model.predict(signals[FEATURES])
    result["mean_baseline"] = train.target.mean()
    result["validation_year"] = year
    predictions.append(result)
    models[year] = model
    split_records.append({
        "validation_year": year,
        "fit_before_signal": signals.date.min(),
        "last_training_target_end": train.label_end.max(),
        "training_rows": len(train),
        "forecast_rows": len(signals),
        "scored_rows": signals.target.notna().sum(),
    })

forecasts = pd.concat(predictions, ignore_index=True)
split_summary = pd.DataFrame(split_records)
assert forecasts.prediction.notna().all()
display(split_summary)

,validation_year,fit_before_signal,last_training_target_end,training_rows,forecast_rows,scored_rows
0,2013,2013-01-02,2012-12-31,20034,7308,7308
1,2014,2014-01-02,2013-12-31,27284,7243,7243
2,2015,2015-01-02,2014-12-31,34590,7027,5353


## 8. Read the model results
Score only rows with an observed target. Show the scored fraction as well.
Missing outcomes can affect the comparison. In 2015, the data end prevents
late signals from completing. These forecasts remain in the saved output.

| Measure | Meaning |
|---|---|
| RMSE | Error measure that gives larger errors more weight. Lower is better. |
| MAE | Mean absolute forecast error. Lower is better. |
| R² | Fit relative to the scored sample's mean return. It can be negative. |
| Mean daily rank IC | Average same-date Spearman correlation between predicted and actual returns. Higher is better. |

An RMSE of `0.20` means 20 percentage points. A constant forecast has no stock
ranking, so its rank IC is undefined. Compute IC only on dates with at least
five observed outcomes and nonconstant predictions and targets.
Daily 90-day targets overlap. Do not interpret these rows as independent trades.
These scores do not measure fees, sector constraints, cash use, or portfolio returns.

In [8]:
metric_rows = []
for year, group in forecasts.groupby("validation_year"):
    scored = group.loc[group.target.notna()]
    for name, column in [("Ridge", "prediction"), ("Training mean", "mean_baseline")]:
        scores = prediction_metrics(scored, scored[column]).to_dict()
        metric_rows.append({"year": year, "model": name, **scores,
                            "scored_fraction": len(scored) / len(group)})
metrics = pd.DataFrame(metric_rows)
display(metrics.round(4))
comparison = metrics.pivot(index="year", columns="model", values="RMSE")
for year, row in comparison.iterrows():
    direction = "lower" if row["Ridge"] < row["Training mean"] else "higher or equal"
    print(f"{year}: Ridge RMSE is {direction} than the training-mean baseline.")

2013: Ridge RMSE is lower than the training-mean baseline.
2014: Ridge RMSE is higher or equal than the training-mean baseline.
2015: Ridge RMSE is lower than the training-mean baseline.


,year,model,RMSE,MAE,R2,mean_daily_rank_IC,IC_dates,scored_fraction
0,2013,Ridge,0.1374,0.0915,0.0891,0.0092,252.0,1.0000
1,2013,Training mean,0.1440,0.0917,-0.0004,NaN,0.0,1.0000
2,2014,Ridge,0.1528,0.0950,-0.1949,-0.0500,252.0,1.0000
3,2014,Training mean,0.1414,0.0894,-0.0225,NaN,0.0,1.0000
4,2015,Ridge,0.1846,0.1135,0.0023,0.2043,189.0,0.7618
5,2015,Training mean,0.1867,0.1103,-0.0203,NaN,0.0,0.7618


## 9. Inspect a forecast ranking
Show the latest signal date in the development forecasts. This preview sorts
forecasts only. It is not an order list. The backtester must apply sector and
stock limits, available cash, and the purchase schedule before placing orders.

Do not require a completed target to produce this ranking. That would use future data.

In [9]:
latest_signal = forecasts.date.max()
ranking = forecasts.loc[forecasts.date.eq(latest_signal)].sort_values(
    ["prediction", "symbol"], ascending=[False, True]
).copy()
ranking["rank"] = np.arange(1, len(ranking) + 1)
display(ranking[["rank", "symbol", "date", "entry_date", "prediction"]].head(5))
print(f"Forecasts on {latest_signal.date()}: {len(ranking)}; completed targets: {ranking.target.notna().sum()}")

Forecasts on 2015-12-30: 27; completed targets: 0


,rank,symbol,date,entry_date,prediction
21553,1,APDN,2015-12-30,2015-12-31,0.243757
21570,2,PEGY,2015-12-30,2015-12-31,0.241187
21560,3,FCX,2015-12-30,2015-12-31,0.119124
21554,4,BESS,2015-12-30,2015-12-31,0.074397
21572,5,SCCO,2015-12-30,2015-12-31,0.054550


## 10. Save small experiment outputs
Save forecasts, validation scores, split dates, and the feature list.
Keep generated outputs outside Git. Do not change the cleaned database.
The saved table contains future audit fields for evaluation. A later allocator
must use only signal dates, stock identifiers, and predictions from this table.

In [10]:
assert file_digest(DATABASE) == database_hash_before
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    forecasts.to_csv(OUTPUT_DIR / "validation_forecasts.csv", index=False)
    metrics.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
    split_summary.to_csv(OUTPUT_DIR / "validation_splits.csv", index=False)
    config = {
        "database": str(DATABASE.relative_to(ROOT)),
        "database_sha256": database_hash_before,
        "data_end": DATA_END, "train_start": TRAIN_START,
        "features": FEATURES, "alpha": RIDGE_ALPHA,
        "validation_years": list(VALIDATION_YEARS),
        "holding_calendar_days": HOLDING_DAYS,
        "max_exit_delay_sessions": MAX_EXIT_DELAY,
    }
    (OUTPUT_DIR / "experiment.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    print(f"Saved outputs: {OUTPUT_DIR.relative_to(ROOT)}")
print("PASS: the cleaned database checksum is unchanged.")

Saved outputs: Homeworks\HW4\Mauricio_Gutierrez\outputs\ridge_baseline\equities_class
PASS: the cleaned database checksum is unchanged.


## Next experiment
1. Read the feature definitions.
2. Review the validation table.
3. Choose one feature or model setting to change.
4. Compare it on the same development dates.
5. Record the decision before opening the final test.

For a new price feature, add its calculation to `price_features` in the Python module.
Add its column name to the notebook's `FEATURES` list. Restart the kernel after
editing an imported module. A financial feature also needs publication-date
alignment; do not join reports on fiscal dates alone.

The loader can read the full cleaned database with the same schema. This notebook
still builds its panel in memory. A full-history run needs a separate memory and
batching plan. Keep development and final-test evaluation separate.

**Sources:** The feature formulas, membership timing, target logic, clipping, and
metrics are adapted from [course_data.py](../../../Lectures/modeling/course_data.py).
The target timing follows [HW4](../../../Lectures/Lecture%208/hw4.md).
Codex helped write and check this first implementation. Continue in `03_backtesting.ipynb` to run the monthly strategy and benchmark comparison.
Run notebook 02 before notebook 03 whenever the features or model settings change.